# YOLO11 fine-tuning on Roboflow pedestrian obstacles


## 1. Install dependencies


In [4]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\taona\Documents\my work\Breast Cancer Histopathology Image Classification\.venv\Scripts\python.exe -m pip install --upgrade pip


## 2. Prepare deterministic train/validation/test split

The downloaded Roboflow export only has a `valid/` folder, so this creates split text files without copying or modifying the dataset images.

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "training"))

from prepare_roboflow_split import OUTPUT_DIR, create_split

create_split()
data_yaml = OUTPUT_DIR / "data.yaml"
data_yaml

Created split at C:\Users\taona\Documents\my work\rutendo_ai\ai_lab\training\roboflow_pedestrian_obstacles_split
train: 2719 images
val: 339 images
test: 341 images


WindowsPath('C:/Users/taona/Documents/my work/rutendo_ai/ai_lab/training/roboflow_pedestrian_obstacles_split/data.yaml')

## 3. Fine-tune YOLO11

Start with `yolo11n.pt` for speed. If accuracy is too weak and the device can handle it, try `yolo11s.pt` later.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_yaml),
    epochs=10,
    imgsz=640,
    batch=-1,
    project=str(PROJECT_ROOT / "training" / "runs"),
    name="yolo11n_roboflow_obstacles",
    patience=15,
    plots=True,
)

Ultralytics 8.4.87  Python-3.12.4 torch-2.12.1+cpu CPU (AMD Ryzen 7 PRO 4750U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\taona\Documents\my work\rutendo_ai\ai_lab\training\roboflow_pedestrian_obstacles_split\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_

## 4. Validate the best checkpoint

In [ ]:
best_model_path = PROJECT_ROOT / "training" / "runs" / "yolo11n_roboflow_obstacles" / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(data_yaml), split="test")
best_model_path

## 5. Quick inference on one image

Use this for visual sanity checks before exporting or wiring the model into the app pipeline.

In [ ]:
sample_image = next((PROJECT_ROOT / "datasets" / "Pedestrian Obstacle Detection.v6-for-validation.yolov11" / "valid" / "images").glob("*.jpg"))
predictions = best_model.predict(str(sample_image), imgsz=640, conf=0.25)
predictions[0].show()

## 6. Use the trained model in the prototype pipeline

In [ ]:
print(f"python src/prototype_detection.py -i videos/test.mp4 --model {best_model_path}")